# Phase 4: Features for every field

Phase 3 gave us checked rasters. This notebook turns them into **one table**: one row per field (1,202), one column per feature. Phase 6 trains the models on this table.

It goes "beyond point analysis": instead of one average per field, it also uses the spread of the pixels inside the field, the texture (GLCM), the field compared with its surroundings, the shape of the season, and the weather in the crop's own growth stages.

**Feature groups**

| Group | What | From |
|---|---|---|
| ground | the 11 features of Phase 5 (year, variety, irrigation, 4 soil groups, sowing day, duration, area, soil-lab flag) | field survey |
| s2_month_mean, s2_month_spread | 7 indices × 6 months: mean, and spread (SD, 10th and 90th percentile) of the pixels | Sentinel-2 |
| s2_season | peak, peak month and window mean of NDVI, NDRE and LSWI in the soybean window (July to October); NDVI gain from June | Sentinel-2 |
| s1_month_mean, s1_month_spread | VV, VH, VH/VV, RVI × 6 months: mean and spread | Sentinel-1 (Lee sigma) |
| s1_season | maximum, window mean, peak month, gain from June and drop to November of VV, VH and VH/VV | Sentinel-1 |
| glcm_s1, glcm_s2 | texture inside the field: VH in July, August, September; NDVI in September | Sentinel-1, Sentinel-2 |
| neighbourhood | field minus its 30 m surroundings: VH in August and September, NDVI in September | Sentinel-1, Sentinel-2 |
| weather_month, era5_rain, rain_month | monthly temperature, maximum temperature, VPD, soil moisture, evaporation; ERA5 rain; CHIRPS rain | ERA5-Land, CHIRPS |
| weather_stage | rain, evaporation, temperature, VPD and soil moisture in 3 crop stages counted from each field's sowing date | ERA5-Land, CHIRPS |
| terrain, soil | elevation, slope, TWI, northness, eastness; clay, sand, organic carbon, pH (0 to 30 cm) | SRTM, MERIT Hydro, SoilGrids |
| modis | LAI, FPAR, GPP, net photosynthesis × 6 months (kept for the Phase 7 physics model) | MODIS |
| s1_quality, s2_quality | images per month, clear looks, share of pixels with a value | both |

**What this notebook does not do:** it trains no model and drops no feature. Dropping features because of their link with yield would be a decision based on yield; Phase 6 tests the **groups** with the Phase 5 folds instead. The 183 hold-out fields get features like every field, but are never used in any check here.

**Before you run:** all Phase 3 rasters, including the 24 `S1_..._leesigma_fields_10m.tif` files, must be in `data/raster/`. Step 3 takes the longest (it reads 48 field files).

## Step 0: Setup and file check

Loads the fields and the hold-out list, and checks that the 62 rasters needed here are present (24 Sentinel-2, 24 Sentinel-1, 4 MODIS, 4 ERA5-Land, 4 CHIRPS, terrain, soil).

**Found:** _(fill after running)_

In [ ]:
from pathlib import Path
import re, warnings, calendar
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window, from_bounds
from scipy import stats
warnings.filterwarnings("ignore")

here = Path.cwd()
ROOT = here if (here / "data" / "raw").exists() else here.parent
RASTER = ROOT / "data" / "raster"
DATA2 = ROOT / "data" / "processed_v2"
OUT = ROOT / "results" / "phase4"; OUT.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "reports" / "figures" / "phase4"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.family": "Times New Roman", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

FILTER = "leesigma"                                   # chosen in P3b Step 4
YEARS, MONTHS = [2022, 2023, 2024, 2025], [6, 7, 8, 9, 10, 11]
S2_BANDS = ["NDVI", "EVI", "GCVI", "LSWI", "NDRE", "NIRv", "RECI", "n_clear"]
S1_BANDS = ["VV", "VH", "VH_VV", "RVI", "NDPI", "n_images"]
CAT = ["year", "variety", "irrigation", "soil_color_group", "soil_texture_group", "soil_depth_group", "soil_structure_group"]
NUM = ["sowing_doy", "crop_duration", "area_acres", "has_soil_lab"]

f = gpd.read_file(DATA2 / "yield_spatial_v2.gpkg")[["uid", "year", "geometry"] + CAT[1:] + NUM]   # 1,202 fields, EPSG:32643
holdout = set(pd.read_csv(ROOT / "results" / "phase5" / "holdout_v2.csv")["uid"])
f["holdout"] = f["uid"].isin(holdout)
print("fields:", len(f), "| per year:", f["year"].value_counts().sort_index().to_dict(), "| hold-out:", int(f["holdout"].sum()))

def path(stem): p = RASTER / f"{stem}.tif"; return p if p.exists() else None
need = ([f"S2_{y}_{m:02d}_fields_10m" for y in YEARS for m in MONTHS] + [f"S1_{y}_{m:02d}_{FILTER}_fields_10m" for y in YEARS for m in MONTHS] +
        [f"{s}_{y}_{r}" for y in YEARS for s, r in [("MODIS", "500m"), ("ERA5", "10km"), ("CHIRPS", "5km")]] + ["DEM_terrain_30m", "SOILGRIDS_250m"])
miss = [s for s in need if path(s) is None]
print(f"rasters found: {len(need) - len(miss)} of {len(need)}" + (" | missing: " + ", ".join(miss) if miss else ""))

## Step 1: Which pixels belong to each field

For each field, the 10 m pixels whose centre is inside the field are found. Pixels on the border are partly the neighbour's land (mixed pixels), so the outline is first **shrunk by 10 m** and only the inner pixels are used, as in v1. If fewer than 5 pixels are left (very small or narrow fields), the whole outline is used. A **ring** of 0 to 30 m outside the field is also kept, for the neighbourhood feature. All Sentinel-1 and Sentinel-2 field files share one grid, so this is done once. Saved: `P4_pixel_counts.csv`.

**Found:** _(fill after running)_

In [ ]:
# Step 1: which 10 m pixels belong to each field. All Sentinel-1 and Sentinel-2 field files share one grid,
# so the pixel masks are made once and reused for every file.
ref = next(path(s) for s in need if s.startswith(("S2_", "S1_")) and path(s))
with rasterio.open(ref) as src: T, SHAPE = src.transform, (src.height, src.width)
MIN_INNER = 5                                          # fewer inner pixels than this: use the whole field instead
masks, rows = {}, []
for _, r in f.iterrows():
    g = r.geometry
    win = from_bounds(*g.buffer(40).bounds, transform=T).round_offsets().round_lengths()   # field + 40 m, so the 30 m ring fits
    win = win.intersection(Window(0, 0, SHAPE[1], SHAPE[0]))                                 # stay inside the file (reading is faster)
    wt = rasterio.windows.transform(win, T); shp = (int(win.height), int(win.width))
    full = rasterize([(g, 1)], out_shape=shp, transform=wt, dtype="uint8") == 1               # pixel centre inside the field
    inner = rasterize([(g.buffer(-10), 1)], out_shape=shp, transform=wt, dtype="uint8") == 1 if not g.buffer(-10).is_empty else np.zeros(shp, bool)
    ring = (rasterize([(g.buffer(30), 1)], out_shape=shp, transform=wt, dtype="uint8") == 1) & ~full   # 0 to 30 m outside the field
    use = inner if inner.sum() >= MIN_INNER else full
    masks[r["uid"]] = (win, use, ring)
    rows.append({"uid": r["uid"], "year": r["year"], "px_field": int(full.sum()), "px_inner": int(inner.sum()), "px_used": int(use.sum()),
                 "used": "inner" if use is inner else "whole field"})
px = pd.DataFrame(rows); px.to_csv(OUT / "P4_pixel_counts.csv", index=False)
print(px.groupby("year")[["px_field", "px_inner", "px_used"]].median().astype(int).to_string())
print("fields using the whole outline (too small after the 10 m shrink):", int((px["used"] == "whole field").sum()), "of", len(px))

## Step 2: The tools

- **Pixel statistics:** mean, standard deviation (SD), 10th and 90th percentile of the field's pixels. The mean says how much; SD and the percentiles say how **even** the field is (a patchy crop has a big spread). A field-month needs values on at least half of its pixels, otherwise it stays empty (clouds).
- **GLCM texture** (Haralick et al. 1973): describes the **pattern** of the pixels, not their level. The values are cut into 16 levels with fixed ranges (VH −25 to −8 dB, NDVI 0 to 1), so texture is comparable between fields. Neighbouring pixel pairs are counted in 4 directions and averaged. Four measures: **contrast** (big jumps between neighbours), **homogeneity** (neighbours alike), **entropy** (disorder) and **correlation** (smooth gradients). Only pairs with both pixels inside the field count, and at least 80% of the field must have values. Honest limit: a field has only about 50 inner pixels, so texture is noisy.
- **Neighbourhood:** field mean minus the mean of the 30 m ring around it. Positive = the field is brighter or greener than what surrounds it.

_No results; this cell only defines the functions._

In [ ]:
# Step 2: the tools. Pixel statistics, GLCM texture and the contrast with the surroundings
STATS = ["mean", "sd", "p10", "p90"]
MIN_VALID = 0.5                                        # a field-month needs values on at least half of its pixels

def pixel_stats(v):
    """mean, standard deviation and 10th / 90th percentile of the pixel values of one field"""
    return [v.mean(), v.std(ddof=1) if v.size > 1 else np.nan, *np.percentile(v, [10, 90])]

GLCM_RANGE = {"VH": (-25.0, -8.0), "NDVI": (0.0, 1.0)}  # fixed ranges, so texture is comparable between fields
LEVELS = 16
def glcm(a, valid, lo, hi):
    """Grey-level co-occurrence matrix of the pixels inside the field (Haralick et al. 1973).
    Values are cut into 16 levels; pairs of neighbouring pixels are counted in 4 directions (0, 45, 90, 135 degrees),
    only when both pixels are inside the field; the 4 results are averaged so the texture does not depend on direction."""
    q = np.clip(((a - lo) / (hi - lo) * LEVELS).astype(int), 0, LEVELS - 1)
    i_, j_ = np.meshgrid(np.arange(LEVELS), np.arange(LEVELS), indexing="ij")
    out, H, W = [], *a.shape
    for dr, dc in [(0, 1), (1, 1), (1, 0), (1, -1)]:
        r0, r1 = 0, H - dr; c0, c1 = max(0, -dc), W - max(0, dc)
        A, B = q[r0:r1, c0:c1], q[r0 + dr:r1 + dr, c0 + dc:c1 + dc]
        ok = valid[r0:r1, c0:c1] & valid[r0 + dr:r1 + dr, c0 + dc:c1 + dc]
        if ok.sum() < 20: continue
        P = np.zeros((LEVELS, LEVELS)); np.add.at(P, (A[ok], B[ok]), 1); P = P + P.T; P /= P.sum()
        mu = (i_ * P).sum(); sd = np.sqrt((((i_ - mu) ** 2) * P).sum())
        nz = P[P > 0]
        out.append([(((i_ - j_) ** 2) * P).sum(),                          # contrast: big jumps between neighbours
                    (P / (1 + (i_ - j_) ** 2)).sum(),                      # homogeneity: neighbours alike
                    -(nz * np.log2(nz)).sum(),                             # entropy: how disordered
                    (((i_ - mu) * (j_ - mu) * P).sum() / sd ** 2) if sd > 0 else np.nan])   # correlation: smooth gradients
    return np.nanmean(out, axis=0) if out else [np.nan] * 4
GLCM_PROPS = ["contrast", "homogeneity", "entropy", "correlation"]
GLCM_DO = {("S1", "VH"): [7, 8, 9], ("S2", "NDVI"): [9]}       # key months: radar in the monsoon and at the peak, NDVI at the peak
RING_DO = {("S1", "VH"): [8, 9], ("S2", "NDVI"): [9]}          # field minus its 30 m surroundings
print("tools ready")

## Step 3: Sentinel-2 and Sentinel-1 features for every field and month

Each of the 48 field files (4 years × 6 months × 2 satellites) is opened once, and every field of that year is read. For each band, the pixel statistics of Step 2 are stored; for the key months also the GLCM texture and the neighbourhood difference.

- Sentinel-2 bands: NDVI, EVI, GCVI, LSWI, NDRE, NIRv, RECI, plus the clear looks (`n_clear`) and the share of pixels with a value.
- Sentinel-1 bands: VV, VH, VH/VV (dB) and RVI, plus the number of images. **NDPI is left out because NDPI = 1 − RVI / 2 exactly**, so it would be a copy.

This is the slowest step; it prints one line per file.

**Found:** _(fill after running)_

In [ ]:
# Step 3: read every Sentinel-2 and Sentinel-1 field file once and compute the field features
feat = {u: {} for u in f["uid"]}
uids_by_year = {y: f.loc[f["year"] == y, "uid"].tolist() for y in YEARS}
jobs = [("S2", f"S2_{{y}}_{{m:02d}}_fields_10m", ["NDVI", "EVI", "GCVI", "LSWI", "NDRE", "NIRv", "RECI"], "n_clear"),
        ("S1", f"S1_{{y}}_{{m:02d}}_{FILTER}_fields_10m", ["VV", "VH", "VH_VV", "RVI"], "n_images")]   # NDPI = 1 - RVI / 2 exactly, so it is left out
for sensor, pattern, bands, count_band in jobs:
    for y in YEARS:
        for m in MONTHS:
            p = path(pattern.format(y=y, m=m))
            if p is None: continue
            with rasterio.open(p) as src:
                names = list(src.descriptions) if all(src.descriptions) else (S2_BANDS if sensor == "S2" else S1_BANDS)
                for u in uids_by_year[y]:
                    win, use, ring = masks[u]
                    a = src.read(window=win).astype(float)
                    d = feat[u]; tag = f"{m:02d}"
                    cnt = a[names.index(count_band)][use]
                    d[f"{sensor.lower()}_{'nclear' if sensor == 'S2' else 'nimg'}_{tag}"] = np.nanmean(cnt) if cnt.size else np.nan
                    first = a[names.index(bands[0])]
                    share = np.isfinite(first[use]).mean() if use.sum() else 0
                    if sensor == "S2": d[f"s2_valid_{tag}"] = share
                    for b in bands:
                        band = a[names.index(b)]; v = band[use]; v = v[np.isfinite(v)]
                        vals = pixel_stats(v) if (share >= MIN_VALID and v.size >= 3) else [np.nan] * 4
                        for s, val in zip(STATS, vals): d[f"{sensor.lower()}_{b}_{tag}_{s}"] = val
                        if m in GLCM_DO.get((sensor, b), []):
                            lo, hi = GLCM_RANGE[b]; ok = use & np.isfinite(band)
                            g = glcm(np.nan_to_num(band, nan=lo), ok, lo, hi) if ok.sum() >= 0.8 * use.sum() else [np.nan] * 4
                            for s, val in zip(GLCM_PROPS, g): d[f"glcm_{b}_{tag}_{s}"] = val
                        if m in RING_DO.get((sensor, b), []):
                            rv = band[ring]; rv = rv[np.isfinite(rv)]
                            d[f"nbr_{b}_{tag}_diff"] = (v.mean() - rv.mean()) if (v.size >= 3 and rv.size >= 5 and share >= MIN_VALID) else np.nan
            print(f"{p.name}: {len(uids_by_year[y])} fields")
F = pd.DataFrame.from_dict(feat, orient="index").reindex(f["uid"]); F.index.name = "uid"   # fields with no file keep an empty row
print("field and month features so far:", F.shape[1])

## Step 4: Season features

From the monthly field means:
- **Sentinel-2**, inside the soybean window July to October (November greenness is other crops, Phase 3): for NDVI, NDRE and LSWI the **peak**, the **month of the peak**, the **window mean** and how many months had a value. Plus the **NDVI gain** (peak minus June, the green-up from bare soil) and the clear looks in the window.
- **Sentinel-1**, the same window: for VV, VH and VH/VV the **maximum**, the **window mean**, the **peak month**, the **gain** (maximum minus June: crop growth) and the **drop** (maximum minus November: harvest). Radar has no cloud gaps, so these exist for every field.

This uses the monthly medians. If the Sentinel-2 group does not help in Phase 6, the next try is a per-image NDVI table with a fitted curve (Jin et al. 2019).

**Found:** _(fill after running)_

In [ ]:
# Step 4: season features from the monthly field means
WIN = [7, 8, 9, 10]                                    # soybean window: July to October (November greenness is other crops)
season = pd.DataFrame(index=F.index)
def col(pfx, b, m): c = f"{pfx}_{b}_{m:02d}_mean"; return F[c] if c in F else pd.Series(np.nan, index=F.index)
for b in ["NDVI", "NDRE", "LSWI"]:                     # Sentinel-2: how green / how much chlorophyll / leaf water at the best moment
    M = pd.concat([col("s2", b, m) for m in WIN], axis=1); M.columns = WIN
    season[f"s2_{b}_peak"] = M.max(axis=1)
    season[f"s2_{b}_peakmonth"] = M.fillna(-np.inf).idxmax(axis=1).where(M.notna().any(axis=1))
    season[f"s2_{b}_winmean"] = M.mean(axis=1)
    season[f"s2_{b}_nmonths"] = M.notna().sum(axis=1)
season["s2_NDVI_gain"] = season["s2_NDVI_peak"] - col("s2", "NDVI", 6)        # green-up from bare soil in June
season["s2_nclear_win"] = sum(F.get(f"s2_nclear_{m:02d}", 0) for m in WIN)     # clear looks in the window (trust)
for b in ["VV", "VH", "VH_VV"]:                       # Sentinel-1: growth and harvest seen through clouds
    M = pd.concat([col("s1", b, m) for m in WIN], axis=1); M.columns = WIN
    season[f"s1_{b}_max"] = M.max(axis=1)
    season[f"s1_{b}_winmean"] = M.mean(axis=1)
    season[f"s1_{b}_peakmonth"] = M.fillna(-np.inf).idxmax(axis=1).where(M.notna().any(axis=1))
    season[f"s1_{b}_gain"] = M.max(axis=1) - col("s1", b, 6)                    # peak minus June (crop growth)
    season[f"s1_{b}_drop"] = M.max(axis=1) - col("s1", b, 11)                   # peak minus November (harvest)
F = F.join(season)
print("season features:", season.shape[1])
print(season[["s2_NDVI_peak", "s2_NDVI_peakmonth", "s1_VH_gain", "s1_VH_peakmonth"]].describe().round(2).to_string())

## Step 5: Weather, rain, MODIS, terrain and soil

- **ERA5-Land (10 km), CHIRPS (5 km), MODIS (500 m), SoilGrids (250 m):** the value of the pixel at a point inside each field. These pixels are much bigger than a field, so **many neighbouring fields share the same value** (printed below). Phase 6 must show whether they help in the leave-one-taluka-out test, where such shared values cannot simply mark a place.
- **Terrain (30 m):** the mean over the field. Aspect is a direction (0° and 360° are the same), so it becomes **northness** (cos) and **eastness** (sin).
- **Soil:** depth-weighted mean of 0 to 30 cm (layers 5, 10 and 15 cm thick), converted to %, g/kg and pH.

**Found:** _(fill after running)_

In [ ]:
# Step 5: weather, rain, MODIS, terrain and soil for every field
pt = f.set_index("uid").geometry.representative_point()                      # a point inside each field
def sample(stem, uids):
    """values of every band at the field points (coarse layers: one pixel per field)"""
    with rasterio.open(path(stem)) as src:
        vals = np.array(list(src.sample([(pt[u].x, pt[u].y) for u in uids]))).astype(float)
        if src.nodata is not None: vals[vals == src.nodata] = np.nan
        return pd.DataFrame(vals, index=uids, columns=list(src.descriptions))
parts = []
for y in YEARS:
    us = uids_by_year[y]; d = pd.DataFrame(index=us)
    for stem, pfx in [(f"ERA5_{y}_10km", "era5"), (f"CHIRPS_{y}_5km", "chirps"), (f"MODIS_{y}_500m", "modis")]:
        if path(stem) is None: continue
        s = sample(stem, us); s.columns = [f"{pfx}_{c}" for c in s.columns]; d = d.join(s)
    parts.append(d)
W = pd.concat(parts); W.index.name = "uid"

# terrain: mean over the field (30 m pixels); aspect is a direction, so it becomes northness and eastness
rows = []
with rasterio.open(path("DEM_terrain_30m")) as src:
    names = list(src.descriptions)
    for _, r in f.iterrows():
        win = from_bounds(*r.geometry.buffer(30).bounds, transform=src.transform).round_offsets().round_lengths()
        a = src.read(window=win, boundless=True, fill_value=np.nan).astype(float)
        m = rasterize([(r.geometry, 1)], out_shape=a.shape[1:], transform=src.window_transform(win), all_touched=True) == 1
        v = {n: a[i][m] for i, n in enumerate(names)}
        asp = np.deg2rad(v["ASPECT"])
        rows.append({"uid": r["uid"], "terrain_ELEV": np.nanmean(v["ELEV"]), "terrain_SLOPE": np.nanmean(v["SLOPE"]),
                     "terrain_TWI": np.nanmean(v["TWI"]), "terrain_northness": np.nanmean(np.cos(asp)), "terrain_eastness": np.nanmean(np.sin(asp))})
TER = pd.DataFrame(rows).set_index("uid")

# soil: depth-weighted mean of 0 to 30 cm (layers 5, 10 and 15 cm thick), in the usual units
s = sample("SOILGRIDS_250m", f["uid"].tolist())
SOIL = pd.DataFrame(index=s.index)
for p_, scale, unit in [("clay", 0.1, "%"), ("sand", 0.1, "%"), ("soc", 0.1, "g/kg"), ("phh2o", 0.1, "pH")]:
    SOIL[f"soil_{p_}"] = (5 * s[f"{p_}_0-5cm_mean"] + 10 * s[f"{p_}_5-15cm_mean"] + 15 * s[f"{p_}_15-30cm_mean"]) / 30 * scale
F = F.join(W).join(TER).join(SOIL)
print("weather / MODIS:", W.shape[1], "| terrain:", TER.shape[1], "| soil:", SOIL.shape[1])
print("fields sharing one ERA5 value (same 10 km pixel), median:", int(W.groupby([c for c in W.columns if c.startswith("era5_T_06")] or [W.index]).size().median()) if "era5_T_06" in W else "")

## Step 6: Weather in the crop's own stages

Fields were sown on different days (mid June to early August), so "July rain" means a young crop for one field and a flowering crop for another. Here the weather is counted from **each field's sowing date**, in three approximate soybean stages (Fehr and Caviness 1977):

| Stage | Days after sowing |
|---|---|
| vegetative (`veg`) | 0 to 35 |
| flowering and pod setting (`flower`) | 35 to 70 |
| seed filling (`fill`) | 70 to 100 |

Monthly totals (rain, actual and potential evaporation) are spread evenly over the days of the month and added up per stage; monthly means (temperature, maximum temperature, VPD, soil moisture) are averaged over the stage days. A stage stays empty if more than 20% of its days fall outside June to November, and for the fields without a sowing date.

**Found:** _(fill after running)_

In [ ]:
# Step 6: weather in the crop's own stages, counted from each field's sowing date
# (approximate soybean stages, Fehr and Caviness 1977): vegetative 0 to 35 days after sowing,
# flowering and pod setting 35 to 70 days, seed filling 70 to 100 days
STAGES = {"veg": (0, 35), "flower": (35, 70), "fill": (70, 100)}
SUMS, MEANS = ["chirps_RAIN", "era5_AET", "era5_PET"], ["era5_T", "era5_TMAX", "era5_VPD", "era5_SM"]
sow = f.set_index("uid")[["year", "sowing_doy"]]
rows = []
for u, r in sow.iterrows():
    out = {"uid": u}
    for st, (a, b) in STAGES.items():
        if pd.isna(r["sowing_doy"]):
            for v in SUMS + MEANS: out[f"stage_{v.split('_')[1]}_{st}"] = np.nan
            continue
        days = pd.Timestamp(int(r["year"]), 1, 1) + pd.to_timedelta(np.arange(int(r["sowing_doy"]) - 1 + a, int(r["sowing_doy"]) - 1 + b), unit="D")
        mon = days.month
        for v in SUMS + MEANS:
            cols = [f"{v}_{m:02d}" for m in mon]
            vals = np.array([F.at[u, c] if c in F.columns else np.nan for c in cols], float)
            if v in SUMS: vals = vals / np.array([calendar.monthrange(int(r["year"]), m)[1] for m in mon])   # monthly total spread over its days
            ok = np.isfinite(vals)
            out[f"stage_{v.split('_')[1]}_{st}"] = (vals[ok].sum() if v in SUMS else vals[ok].mean()) if ok.mean() >= 0.8 else np.nan
    rows.append(out)
STG = pd.DataFrame(rows).set_index("uid")
F = F.join(STG)
print("crop-stage weather features:", STG.shape[1], "| fields without a sowing date:", int(sow["sowing_doy"].isna().sum()))
print(STG[[c for c in STG.columns if c.startswith("stage_RAIN")]].describe().round(0).to_string())

## Step 7: Ground data and the feature list

Adds the 11 ground features of Phase 5 (prefix `ground_`), then gives every feature a **group** and a **source**, and counts missing values per group. The groups are what Phase 6 will test one at a time.

**Found:** _(fill after running)_

In [ ]:
# Step 7: the ground data (the same 11 features as Phase 5) and the feature list with groups
G = f.set_index("uid")[CAT + NUM].copy()
G.columns = [f"ground_{c}" if c != "year" else "year" for c in G.columns]
X = f.set_index("uid")[["holdout"]].join(G).join(F)
def group(c):
    if c in ("year",) or c.startswith("ground_"): return "ground"
    if re.match(r"s2_(nclear|valid)_", c) or c in ("s2_nclear_win",) or c.endswith("_nmonths"): return "s2_quality"
    if re.match(r"s2_\w+_\d\d_mean$", c): return "s2_month_mean"
    if re.match(r"s2_\w+_\d\d_(sd|p10|p90)$", c): return "s2_month_spread"
    if c.startswith("s2_"): return "s2_season"
    if c.startswith("s1_nimg_"): return "s1_quality"
    if re.match(r"s1_\w+_\d\d_mean$", c): return "s1_month_mean"
    if re.match(r"s1_\w+_\d\d_(sd|p10|p90)$", c): return "s1_month_spread"
    if c.startswith("s1_"): return "s1_season"
    if c.startswith("glcm_VH"): return "glcm_s1"
    if c.startswith("glcm_NDVI"): return "glcm_s2"
    if c.startswith("nbr_"): return "neighbourhood"
    if c.startswith("era5_PRECIP"): return "era5_rain"
    if c.startswith("era5_"): return "weather_month"
    if c.startswith("chirps_"): return "rain_month"
    if c.startswith("stage_"): return "weather_stage"
    if c.startswith("modis_"): return "modis"
    if c.startswith("terrain_"): return "terrain"
    if c.startswith("soil_"): return "soil"
    return "other"
SRC = {"ground": "field survey", "s2": "Sentinel-2 SR + Cloud Score+", "s1": f"Sentinel-1 GRD, orbit 63, {FILTER}", "glcm_s1": "Sentinel-1 VH", "glcm_s2": "Sentinel-2 NDVI",
       "neighbourhood": "Sentinel-1 / Sentinel-2, 30 m ring", "era5": "ERA5-Land", "chirps": "CHIRPS v2.0", "stage": "ERA5-Land / CHIRPS + sowing date",
       "modis": "MODIS MOD15A2H / MOD17A2HGF", "terrain": "SRTM + MERIT Hydro", "soil": "SoilGrids 2.0"}
feats = [c for c in X.columns if c != "holdout"]
dic = pd.DataFrame({"feature": feats, "group": [group(c) for c in feats]})
dic["source"] = [next((v for k, v in SRC.items() if c.startswith(k) or g.startswith(k)), "") for c, g in zip(dic["feature"], dic["group"])]
dic["missing_%"] = [round(100 * X[c].isna().mean(), 1) for c in feats]
print("features:", len(feats), "| fields:", len(X))
print(dic.groupby("group").agg(features=("feature", "size"), missing_mean_pct=("missing_%", "mean")).round(1).to_string())

## Step 8: Checks

Only the **1,019 development fields** are used; the hold-out stays untouched.

- **(a) Near-copies:** pairs of features with Spearman |rho| above 0.95. They carry the same information (for example a month that is always the peak). Nothing is dropped here; the list helps to read Phase 6 and to simplify the final model.
- **(b) Link with yield:** Spearman rho of each feature with yield, over all development fields and **inside 2024 and inside 2025** (the two big years). Inside one year, the differences between years (a wet year, a good year) cannot create a link, so this is the fairer number. This is a first look, not a selection: with about 440 features, some will look linked by chance.

Saved: `P4_feature_checks.csv` (missing share and rho for every feature).

**Found:** _(fill after running)_

In [ ]:
# Step 8: checks. Missing values, near-copies, and a first look at the link with yield (development fields only)
y = gpd.read_file(DATA2 / "yield_spatial_v2.gpkg").set_index("uid")["crop_yield"]
dev = X.index[~X["holdout"]]                                                  # the 183 hold-out fields are never used here
num = [c for c in feats if c not in ["year"] + [f"ground_{c}" for c in CAT[1:]] and pd.api.types.is_numeric_dtype(X[c])]
D = X.loc[dev, num]
# (a) near-copies: Spearman |rho| above 0.95 between two features (computed on ranks)
R = D.rank().corr().abs(); R = R.where(~np.eye(len(R), dtype=bool), 0)
pairs = R.where(np.triu(np.ones(R.shape, bool), 1)).stack(); pairs = pairs[pairs > 0.95].sort_values(ascending=False)
print(f"(a) pairs of near-copies (|rho| > 0.95): {len(pairs)}; features involved: {len(set(pairs.index.get_level_values(0)) | set(pairs.index.get_level_values(1)))} of {len(num)}")
print(pairs.head(10).round(3).to_string())
# (b) link with yield: Spearman rho over all development fields, and inside 2024 and 2025 separately
#     (inside one year, the year-to-year differences in weather and yield cannot create a link)
def rho(a, b):
    ok = a.notna() & b.notna()
    return stats.spearmanr(a[ok], b[ok])[0] if ok.sum() >= 20 and a[ok].nunique() > 1 else np.nan
yd = y.reindex(dev); yrs = X.loc[dev, "year"]
rows = []
for c in num:
    rows.append({"feature": c, "n": int(D[c].notna().sum()), "rho_all": rho(D[c], yd),
                 "rho_2024": rho(D[c][yrs == 2024], yd[yrs == 2024]), "rho_2025": rho(D[c][yrs == 2025], yd[yrs == 2025])})
link = pd.DataFrame(rows).merge(dic[["feature", "group"]])
link["rho_within_mean"] = link[["rho_2024", "rho_2025"]].mean(axis=1)
chk = dic.merge(link, how="left"); chk.to_csv(OUT / "P4_feature_checks.csv", index=False)
print("\n(b) strongest link with yield inside a year (mean of 2024 and 2025 rho), top 15:")
print(link.reindex(link["rho_within_mean"].abs().sort_values(ascending=False).index).head(15).round(3).to_string(index=False))
print("\nbest |rho| inside a year, per group:")
print(link.assign(a=link["rho_within_mean"].abs()).groupby("group")["a"].max().sort_values(ascending=False).round(3).to_string())

## Step 9: Feature maps

Four features for the 2025 fields, one dot per field: the Sentinel-2 NDVI peak, the Sentinel-1 VH gain, rain at flowering, and VH texture in August. Colour scales are cut at the 2nd and 98th percentile. Saved: `P4_feature_maps_2025.png`.

**Found:** _(fill after running)_

In [ ]:
# Step 9: maps of four features for the 2025 fields (one dot per field)
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter
bnd = ROOT / "data" / "boundaries" / "latur_boundaries_2001.gpkg"
district = gpd.read_file(bnd, layer="latur_district").to_crs(32643) if bnd.exists() else None
talukas = gpd.read_file(bnd, layer="latur_talukas").to_crs(32643) if bnd.exists() else None
SHOW = [("s2_NDVI_peak", "Sentinel-2 NDVI peak, July to October", "RdYlGn"), ("s1_VH_gain", "Sentinel-1 VH gain, peak minus June (dB)", "viridis"),
        ("stage_RAIN_flower", "Rain at flowering, 35 to 70 days after sowing (mm)", "YlGnBu"), ("glcm_VH_08_homogeneity", "VH texture, August: homogeneity", "magma")]
yr = 2025; u = uids_by_year[yr]; p25 = pt[u]
fig, axes = plt.subplots(2, 2, figsize=(13, 12))
x0, y0, x1, y1 = f.total_bounds
for ax, (c, title, cm) in zip(axes.ravel(), SHOW):
    v = X.loc[u, c] if c in X else pd.Series(np.nan, index=u)
    if talukas is not None: talukas.boundary.plot(ax=ax, color="0.5", lw=0.5)
    if district is not None: district.boundary.plot(ax=ax, color="black", lw=1)
    lo, hi = np.nanpercentile(v, [2, 98]) if v.notna().any() else (0, 1)
    sc = ax.scatter(p25.x, p25.y, c=v, cmap=cm, vmin=lo, vmax=hi, s=12, edgecolor="0.3", lw=0.2)
    plt.colorbar(sc, ax=ax, shrink=0.75)
    ax.set_xlim(x0 - 1000, x1 + 1000); ax.set_ylim(y0 - 1000, y1 + 1000); ax.set_aspect("equal")
    km = FuncFormatter(lambda t, _: f"{t / 1000:.0f}"); ax.xaxis.set_major_formatter(km); ax.yaxis.set_major_formatter(km)
    ax.tick_params(labelsize=8); ax.grid(color="0.6", lw=0.3, ls=":"); ax.set_xlabel("Easting (km)", fontsize=9); ax.set_ylabel("Northing (km)", fontsize=9)
    sx, sy = x0 + 1000, y0
    for k in range(2): ax.add_patch(Rectangle((sx + k * 5000, sy), 5000, 600, fc="black" if k == 0 else "white", ec="black", lw=0.6))
    ax.text(sx, sy + 1100, "0", fontsize=7, ha="center"); ax.text(sx + 10000, sy + 1100, "10 km", fontsize=7, ha="center")
    ax.annotate("N", xy=(0.95, 0.95), xytext=(0.95, 0.85), xycoords="axes fraction", ha="center", fontsize=10, fontweight="bold", arrowprops=dict(arrowstyle="-|>", lw=1.2))
    ax.set_title(f"{title}, {yr}", loc="left", fontsize=11)
fig.text(0.01, 0.0, "Source: Sentinel-2 SR, Sentinel-1 GRD (Lee sigma), CHIRPS v2.0, field survey | WGS 84 / UTM zone 43N (EPSG:32643)", fontsize=9, color="0.3")
fig.tight_layout(); fig.savefig(FIG / "P4_feature_maps_2025.png", dpi=250, facecolor="white", bbox_inches="tight"); plt.show()

## Step 10: Save

- `P4_features.csv`: uid, hold-out flag and all features (no yields; Phase 6 joins them by uid).
- `P4_feature_dictionary.csv`: every feature with its group, source and missing share.
- `P4_feature_checks.csv` and `P4_pixel_counts.csv` from the steps above.

**Found:** _(fill after running)_

In [ ]:
# Step 10: save. One row per field, one column per feature. No yields in this table (Phase 6 joins them by uid)
X.reset_index().to_csv(OUT / "P4_features.csv", index=False)
dic.to_csv(OUT / "P4_feature_dictionary.csv", index=False)
print(f"saved: P4_features.csv ({X.shape[0]} fields x {len(feats)} features), P4_feature_dictionary.csv, P4_feature_checks.csv, P4_pixel_counts.csv")

## Summary

_(fill after running)_

| Question | Answer |
|---|---|
| Features per group | |
| Pixels per field used | |
| Most missing groups | |
| Near-copies | |
| Strongest links inside a year | |

**Next (Phase 6):** train on this table with the Phase 5 folds: ground only, then add one group at a time (Sentinel-2, Sentinel-1, texture, weather, ...). A group stays only if it improves r under leave-one-taluka-out.